# A04 customer_behavior — keras

Mã huấn luyện tách từ notebook gốc, giữ tham số/thuật toán. Mặc định `RUN_TRAINING=False`: Run All chỉ kiểm imports/path và xem số liệu đã lưu; không fit/TEST. Muốn thực nghiệm mới phải tự bật cờ; mọi output legacy path được chuyển vào một thư mục `notebooks/executed/artifacts/` mới, không ghi đè kết quả gốc.


In [ ]:
# Migration r2: select this assignment's original logical paths, preserving frozen source.
from pathlib import Path as _Path
import sys as _sys
_repo = next(p for p in (_Path.cwd(), *_Path.cwd().parents) if (p / 'ASSIGN_REORGANIZATION_EXECUTION_PLAN.md').exists())
_sys.path.insert(0, str(_repo / '_shared'))
from assignment_runtime import activate as _activate
_migration = _activate('A04', 'keras')
_migration


In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    from assignment_runtime import enable_training_output_overlay
    enable_training_output_overlay()
from pathlib import Path
import json
import pandas as pd
PROJECT_ROOT = Path(_migration['logical_root'])


# 1. Assignment Overview

Cell này làm gì?

Notebook này tiếp tục bài toán A03 Customer Behavior: dự đoán customer churn từ hành vi mua hàng và tương tác của khách. A04 dùng Olist Brazilian E-Commerce Public Dataset đã được chuẩn bị thành customer-level churn dataset trong `datasets/customer_behavior/a04`.


In [ ]:
if RUN_TRAINING:
    from pathlib import Path
    import json
    import pandas as pd

    def find_project_root(start=None):
        start = Path.cwd() if start is None else Path(start)
        for path in [start.resolve(), *start.resolve().parents]:
            if (path / "datasets" / "customer_behavior" / "a04").exists() and (path / "pipeline").exists():
                return path
        raise FileNotFoundError("Cannot locate project root from current working directory.")

    PROJECT_ROOT = find_project_root()
    DATA_DIR = PROJECT_ROOT / "datasets" / "customer_behavior" / "a04"
    A03_NOTEBOOK = PROJECT_ROOT / "pipeline" / "customer_behavior_pipeline.ipynb"
    A03_METADATA_PATH = PROJECT_ROOT / "pipeline" / "customer_behavior" / "metadata.json"
    A03_COMPARISON_PATH = PROJECT_ROOT / "pipeline" / "customer_behavior" / "final_comparison.csv"
    A04_METADATA_PATH = DATA_DIR / "metadata.json"

    with A04_METADATA_PATH.open("r", encoding="utf-8") as f:
        dataset_metadata = json.load(f)
    with A03_METADATA_PATH.open("r", encoding="utf-8") as f:
        a03_metadata_preview = json.load(f)

    overview = {
        "A03 business problem": a03_metadata_preview["task_name"],
        "A03 target": a03_metadata_preview["target"],
        "A03 task type": a03_metadata_preview["task_type"],
        "A03 positive label": "is_churned = 1 means churned customer",
        "A04 dataset": dataset_metadata["dataset_name"],
        "A04 source": dataset_metadata["source"],
        "A04 target": dataset_metadata["target"],
        "A04 samples": dataset_metadata["samples"],
        "A04 features": dataset_metadata["features"],
        "Task type": "Binary classification",
    }
    pd.DataFrame([overview]).T.rename(columns={0: "Value"})


Output cho thấy điều gì?

Bảng overview xác nhận A04 giữ đúng business problem của A03: dự đoán `is_churned` cho khách hàng. Dataset A04 là Olist customer-level churn với 58,101 samples, 29 raw input features và target nhị phân; `is_churned=1` nghĩa là khách không có repeat order hợp lệ trong outcome window 180 ngày.


# 2. Environment Setup

Cell này làm gì?

Import thư viện, cố định seed, kiểm tra TensorFlow/PyTorch version và thiết bị chạy. Notebook dùng `pathlib.Path` để tránh hard-code đường dẫn cá nhân.


In [ ]:
import os
import random
import time
import warnings
from datetime import datetime, timezone
import joblib
import matplotlib.pyplot as plt
import numpy as np
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
warnings.filterwarnings('ignore', category=FutureWarning)
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.get_logger().setLevel('ERROR')
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)
ARTIFACT_DIR = PROJECT_ROOT / 'pipeline04' / 'customer_behavior'
MODEL_DIR = ARTIFACT_DIR / 'models'
METRIC_DIR = ARTIFACT_DIR / 'metrics'
FIGURE_DIR = ARTIFACT_DIR / 'figures'
for directory in [ARTIFACT_DIR, MODEL_DIR, METRIC_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
env_info = {'Python executable': Path(os.sys.executable).name, 'NumPy': np.__version__, 'Pandas': pd.__version__, 'scikit-learn': sklearn.__version__, 'TensorFlow': tf.__version__, 'TensorFlow GPUs': len(tf.config.list_physical_devices('GPU')), 'Project root is portable': PROJECT_ROOT.name == 'assign'}
pd.DataFrame([env_info]).T.rename(columns={0: 'Value'})


Output cho thấy điều gì?

Environment có đủ TensorFlow và PyTorch để chạy cùng một prepared customer dataset. CPU vẫn chạy được vì dữ liệu đã aggregate ở cấp customer, không phải event stream thô nhiều triệu dòng.


# 3. Load Dataset

Cell này làm gì?

Liệt kê raw Olist files và đọc prepared customer-level churn CSV. Raw files được giữ nguyên; notebook dùng file prepared để tránh tự tạo target khác A03.


In [ ]:
if RUN_TRAINING:
    all_files = sorted(DATA_DIR.glob("*.csv"))
    file_listing = [
        {"file": path.relative_to(PROJECT_ROOT).as_posix(), "size_mb": round(path.stat().st_size / (1024 * 1024), 2)}
        for path in all_files
    ]
    display(pd.DataFrame(file_listing))

    DATA_PATH = DATA_DIR / "olist_customer_churn_a04.csv"
    TARGET = "is_churned"
    df_raw = pd.read_csv(DATA_PATH)

    print("Loaded dataset:", DATA_PATH.relative_to(PROJECT_ROOT).as_posix())
    print("Shape:", df_raw.shape)
    display(df_raw.head())


Output cho thấy điều gì?

Folder A04 có raw relational Olist tables và một prepared customer-level file. Notebook đọc `olist_customer_churn_a04.csv` vì file này giữ target `is_churned` đã chốt cho continuity với A03.


# 4. Dataset Inspection

Cell này làm gì?

Kiểm tra shape, columns, target distribution, missing values, duplicates, descriptive statistics, categorical columns và evidence về sequence/time structure.


In [ ]:
if RUN_TRAINING:
    shape_info = pd.DataFrame(
        {"Metric": ["Rows", "Columns", "Candidate feature columns"], "Value": [df_raw.shape[0], df_raw.shape[1], df_raw.shape[1] - 2]}
    )
    dtype_info = df_raw.dtypes.astype(str).reset_index()
    dtype_info.columns = ["Column", "Dtype"]
    missing = df_raw.isna().sum().reset_index()
    missing.columns = ["Column", "MissingCount"]
    missing["MissingRate"] = missing["MissingCount"] / len(df_raw)
    duplicate_count = int(df_raw.duplicated().sum())

    target_counts = df_raw[TARGET].value_counts().sort_index()
    target_distribution = pd.DataFrame(
        {
            "Class": target_counts.index.astype(int),
            "Count": target_counts.values,
            "Rate": (target_counts.values / len(df_raw)).round(4),
            "LabelInterpretation": ["not churned / returned in outcome window", "churned / no repeat order in outcome window"],
        }
    )

    categorical_preview = df_raw.select_dtypes(include=["object"]).nunique().reset_index()
    categorical_preview.columns = ["Column", "UniqueValues"]
    numeric_columns = [c for c in df_raw.select_dtypes(include=[np.number]).columns if c != TARGET]

    orders_path = DATA_DIR / "olist_orders_dataset.csv"
    orders_raw = pd.read_csv(orders_path, usecols=["order_id", "customer_id", "order_status", "order_purchase_timestamp"])
    orders_raw["order_purchase_timestamp"] = pd.to_datetime(orders_raw["order_purchase_timestamp"], errors="coerce")
    sequence_evidence = pd.DataFrame(
        [
            {"Representation": "prepared customer CSV", "Evidence": "one row per customer_unique_id with aggregate behavior features", "ModelingUse": "main tabular MLP input"},
            {"Representation": "raw Olist orders", "Evidence": f"{orders_raw.shape[0]:,} timestamped order events from {orders_raw['order_purchase_timestamp'].min()} to {orders_raw['order_purchase_timestamp'].max()}", "ModelingUse": "evidence of temporal source, not used as Conv1D input in this notebook"},
            {"Representation": "clickstream/session sequence", "Evidence": "not present in prepared file or raw Olist tables", "ModelingUse": "no CNN/sequence production model"},
        ]
    )

    display(shape_info)
    display(dtype_info)
    display(missing[missing["MissingCount"] > 0] if missing["MissingCount"].sum() else pd.DataFrame({"Missing": ["No missing values detected"]}))
    print("Duplicate rows:", duplicate_count)
    display(target_distribution)
    display(df_raw[numeric_columns].describe().T.round(3))
    display(categorical_preview)
    display(sequence_evidence)

    fig, ax = plt.subplots(figsize=(5.5, 3.5))
    ax.bar(target_distribution["Class"].astype(str), target_distribution["Count"], color=["#4C78A8", "#F58518"])
    ax.set_title("Olist Customer Churn Label Distribution")
    ax.set_xlabel("is_churned")
    ax.set_ylabel("Count")
    for i, row in target_distribution.iterrows():
        ax.text(i, row["Count"], f'{row["Rate"]:.1%}', ha="center", va="bottom")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "target_churn_distribution.png", dpi=160)
    plt.show()


Output cho thấy điều gì?

Prepared dataset có 58,101 dòng và 31 cột, không có missing value trong file đã chuẩn bị và không có duplicate row. Target cực kỳ mất cân bằng: class 0 có 665 khách (1.14%), class 1 có 57,436 khách (98.86%); vì vậy notebook báo thêm Macro F1 và Balanced Accuracy để tránh bị accuracy đánh lừa.


# 5. Data Preparation

Cell này làm gì?

Loại identifier, group rare categories theo train only, chia train/validation/test bằng stratify, rồi fit imputer/scaler/one-hot encoder chỉ trên training data.


In [ ]:
if RUN_TRAINING:
    identifier_columns = ["customer_unique_id"]
    numeric_features = dataset_metadata["numerical_features"]
    categorical_features = dataset_metadata["categorical_features"]
    feature_cols = numeric_features + categorical_features

    df_model = df_raw.drop_duplicates().reset_index(drop=True)
    X = df_model[feature_cols].copy()
    y = df_model[TARGET].astype(int)

    X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, random_state=SEED, stratify=y
    )
    X_valid_raw, X_test_raw, y_valid, y_test = train_test_split(
        X_temp_raw, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
    )

    top_category_limits = {
        "primary_category": 30,
        "primary_payment_type": 10,
        "customer_city": 80,
        "customer_state": 30,
    }
    category_maps = {}
    for col in categorical_features:
        top_values = X_train_raw[col].fillna("Unknown").value_counts().head(top_category_limits[col]).index.tolist()
        category_maps[col] = set(top_values)

    def apply_rare_category_grouping(frame):
        out = frame.copy()
        for col in categorical_features:
            values = out[col].fillna("Unknown").astype(str)
            out[col] = values.where(values.isin(category_maps[col]), "Other")
        return out

    X_train_raw = apply_rare_category_grouping(X_train_raw)
    X_valid_raw = apply_rare_category_grouping(X_valid_raw)
    X_test_raw = apply_rare_category_grouping(X_test_raw)

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_features,
            ),
            (
                "categorical",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                    ]
                ),
                categorical_features,
            ),
        ],
        sparse_threshold=0.0,
    )

    X_train = preprocessor.fit_transform(X_train_raw).astype("float32")
    X_valid = preprocessor.transform(X_valid_raw).astype("float32")
    X_test = preprocessor.transform(X_test_raw).astype("float32")
    y_train_np = y_train.to_numpy(dtype="float32")
    y_valid_np = y_valid.to_numpy(dtype="float32")
    y_test_np = y_test.to_numpy(dtype="float32")
    transformed_feature_names = preprocessor.get_feature_names_out()

    joblib.dump({"preprocessor": preprocessor, "category_maps": {k: sorted(v) for k, v in category_maps.items()}}, MODEL_DIR / "preprocessor.joblib")

    split_summary = pd.DataFrame(
        [
            {"Split": "train", "Rows": len(y_train), "ChurnRate": y_train.mean(), "NotChurned": int((y_train == 0).sum()), "Churned": int((y_train == 1).sum())},
            {"Split": "validation", "Rows": len(y_valid), "ChurnRate": y_valid.mean(), "NotChurned": int((y_valid == 0).sum()), "Churned": int((y_valid == 1).sum())},
            {"Split": "test", "Rows": len(y_test), "ChurnRate": y_test.mean(), "NotChurned": int((y_test == 0).sum()), "Churned": int((y_test == 1).sum())},
        ]
    )
    preprocess_summary = pd.DataFrame(
        [{"RawFeatures": len(feature_cols), "NumericFeatures": len(numeric_features), "CategoricalFeatures": len(categorical_features), "TransformedFeatures": X_train.shape[1]}]
    )

    display(split_summary.round(4))
    display(preprocess_summary)
    display(pd.DataFrame({"TransformedFeature": transformed_feature_names[:25]}))


Output cho thấy điều gì?

Split được stratify thành train/validation/test và preprocessor chỉ fit trên train set. Sau encoding và scaling, mỗi khách hàng được biểu diễn bằng vector tabular gồm 169 features; target test vẫn được giữ riêng cho final evaluation.


# 6. Deep-Learning Data Representation

Cell này làm gì?

In shape của tensor đầu vào deep learning và đánh giá representation có phù hợp với CNN/sequence modeling không.


In [ ]:
if RUN_TRAINING:
    representation_summary = pd.DataFrame(
        {
            "Array": ["X_train", "y_train", "X_valid", "y_valid", "X_test", "y_test"],
            "Shape": [X_train.shape, y_train_np.shape, X_valid.shape, y_valid_np.shape, X_test.shape, y_test_np.shape],
            "Meaning": [
                "(samples, transformed tabular features)",
                "(samples,)",
                "(samples, transformed tabular features)",
                "(samples,)",
                "(samples, transformed tabular features)",
                "(samples,)",
            ],
        }
    )
    representation_decision = pd.DataFrame(
        [
            {"Question": "Independent tabular features?", "Assessment": "Yes: prepared file is one customer row with aggregate behavioral features"},
            {"Question": "Ordered session/clickstream?", "Assessment": "No: prepared CSV has no event order or clickstream columns"},
            {"Question": "Raw transaction sequence exists?", "Assessment": "Partially: raw Olist order timestamps exist, but the model input selected here is customer-level aggregate"},
            {"Question": "Natural baseline", "Assessment": "MLP/DNN classification"},
            {"Question": "CNN justified?", "Assessment": "No production CNN for prepared representation; only toy Conv1D operation is shown"},
        ]
    )
    display(representation_summary)
    display(representation_decision)


Output cho thấy điều gì?

Representation chính là customer-level aggregate tabular vector, không phải ảnh và cũng chưa phải event sequence. Raw Olist có timestamped orders/payments/reviews, nhưng notebook này không tự reorder columns để tạo sequence giả; MLP/DNN là baseline tự nhiên, còn Conv1D chỉ được minh họa như operation giáo dục.


# 7. Discover Neural Network and CNN Operations

Cell này làm gì?

Minh họa Dense/Linear, ReLU, logits, sigmoid, Conv1D và pooling trên toy tensors.


In [ ]:
if RUN_TRAINING:
    toy_x = tf.constant([[2.0, 0.5, -1.0]], dtype=tf.float32)
    dense = keras.Sequential(
        [
            layers.Input(shape=(3,)),
            layers.Dense(2, use_bias=True, kernel_initializer="ones", bias_initializer="zeros"),
        ]
    )
    toy_logits = dense(toy_x)
    toy_relu = tf.nn.relu(toy_logits)
    toy_sigmoid = tf.nn.sigmoid(toy_logits)

    toy_seq = tf.reshape(tf.range(1, 9, dtype=tf.float32), (1, 8, 1))
    conv = layers.Conv1D(filters=2, kernel_size=3, padding="valid", activation="relu")
    conv_out = conv(toy_seq)
    pool_out = layers.MaxPooling1D(pool_size=2)(conv_out)

    operation_shapes = pd.DataFrame(
        [
            {"Object": "toy_x", "Shape": tuple(toy_x.shape), "Meaning": "one toy customer vector"},
            {"Object": "Dense logits", "Shape": tuple(toy_logits.shape), "Meaning": "linear outputs before probability"},
            {"Object": "ReLU(logits)", "Shape": tuple(toy_relu.shape), "Meaning": "negative values clipped to zero"},
            {"Object": "Sigmoid(logits)", "Shape": tuple(toy_sigmoid.shape), "Meaning": "probability-like values 0..1"},
            {"Object": "Toy sequence", "Shape": tuple(toy_seq.shape), "Meaning": "(batch, steps, channels)"},
            {"Object": "Conv1D output", "Shape": tuple(conv_out.shape), "Meaning": "local windows over ordered toy sequence"},
            {"Object": "Pooling output", "Shape": tuple(pool_out.shape), "Meaning": "downsampled Conv1D features"},
        ]
    )
    display(operation_shapes)
    print("Dense logits:", toy_logits.numpy().round(4))
    print("Sigmoid probabilities:", toy_sigmoid.numpy().round(4))


Output cho thấy điều gì?

Dense tạo logits từ vector customer features, sigmoid chuyển logits thành xác suất churn. Conv1D/pooling chỉ minh họa trên toy sequence có thứ tự rõ, không áp đặt lên các cột aggregate của Olist.


# 8. Neural Network / CNN From Scratch

Cell này làm gì?

Tự implement bằng NumPy các operation giáo dục: linear layer, ReLU, sigmoid, binary cross entropy, forward composition và Conv1D toy.


In [ ]:
if RUN_TRAINING:
    def linear_np(x, w, b):
        return x @ w + b

    def relu_np(x):
        return np.maximum(0, x)

    def sigmoid_np(x):
        return 1.0 / (1.0 + np.exp(-x))

    def binary_cross_entropy_np(y_true, y_prob, eps=1e-7):
        y_prob = np.clip(y_prob, eps, 1 - eps)
        return -np.mean(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))

    def forward_np(x, params):
        z1 = linear_np(x, params["w1"], params["b1"])
        a1 = relu_np(z1)
        z2 = linear_np(a1, params["w2"], params["b2"])
        return sigmoid_np(z2)

    def conv1d_valid_np(sequence, kernel, bias=0.0):
        sequence = np.asarray(sequence, dtype=float)
        kernel = np.asarray(kernel, dtype=float)
        out = []
        for i in range(len(sequence) - len(kernel) + 1):
            out.append(float(np.sum(sequence[i : i + len(kernel)] * kernel) + bias))
        return np.asarray(out)

    rng = np.random.default_rng(SEED)
    toy_batch = X_train[:4]
    params = {
        "w1": rng.normal(0, 0.1, size=(X_train.shape[1], 8)),
        "b1": np.zeros(8),
        "w2": rng.normal(0, 0.1, size=(8, 1)),
        "b2": np.zeros(1),
    }
    toy_probs = forward_np(toy_batch, params).ravel()
    toy_loss = binary_cross_entropy_np(y_train_np[:4], toy_probs)
    toy_conv = conv1d_valid_np([1, 2, 3, 4, 5], [0.25, 0.5, 0.25])

    print("Forward churn probabilities:", np.round(toy_probs, 4))
    print("Toy BCE:", round(float(toy_loss), 4))
    print("Manual Conv1D toy output:", np.round(toy_conv, 4))


Output cho thấy điều gì?

NumPy forward pass minh họa cách MLP tạo xác suất churn từ customer aggregate features. Conv1D thủ công chỉ giải thích operation trên toy sequence, không dùng làm model chính cho prepared Olist table.


# 9. Deep Learning with Keras/TensorFlow

Cell này làm gì?

Định nghĩa helper metric, threshold selection và baseline Keras MLP cho churn classification.


In [ ]:
if RUN_TRAINING:
    def to_jsonable(obj):
        if isinstance(obj, dict):
            return {str(k): to_jsonable(v) for k, v in obj.items()}
        if isinstance(obj, list):
            return [to_jsonable(v) for v in obj]
        if isinstance(obj, tuple):
            return [to_jsonable(v) for v in obj]
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.floating,)):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return obj

    def save_json(path, obj):
        with Path(path).open("w", encoding="utf-8") as f:
            json.dump(to_jsonable(obj), f, indent=2)

    def select_threshold(y_true, prob):
        thresholds = np.round(np.linspace(0.05, 0.95, 91), 2)
        rows = []
        for threshold in thresholds:
            pred = (prob >= threshold).astype(int)
            rows.append(
                {
                    "Threshold": float(threshold),
                    "Accuracy": accuracy_score(y_true, pred),
                    "Precision": precision_score(y_true, pred, zero_division=0),
                    "Recall": recall_score(y_true, pred, zero_division=0),
                    "F1": f1_score(y_true, pred, zero_division=0),
                    "MacroF1": f1_score(y_true, pred, average="macro", zero_division=0),
                    "BalancedAccuracy": balanced_accuracy_score(y_true, pred),
                }
            )
        table = pd.DataFrame(rows)
        best = table.sort_values(["MacroF1", "BalancedAccuracy", "F1"], ascending=False).iloc[0]
        return float(best["Threshold"]), table

    def evaluate_probabilities(y_true, prob, threshold=None):
        if threshold is None:
            threshold, _ = select_threshold(y_true, prob)
        pred = (prob >= threshold).astype(int)
        cm = confusion_matrix(y_true, pred)
        tn, fp, fn, tp = cm.ravel()
        return {
            "Threshold": float(threshold),
            "Accuracy": float(accuracy_score(y_true, pred)),
            "Precision": float(precision_score(y_true, pred, zero_division=0)),
            "Recall": float(recall_score(y_true, pred, zero_division=0)),
            "F1": float(f1_score(y_true, pred, zero_division=0)),
            "MacroF1": float(f1_score(y_true, pred, average="macro", zero_division=0)),
            "BalancedAccuracy": float(balanced_accuracy_score(y_true, pred)),
            "ROC-AUC": float(roc_auc_score(y_true, prob)),
            "TN": int(tn),
            "FP": int(fp),
            "FN": int(fn),
            "TP": int(tp),
        }

    def build_basic_mlp(input_dim, name="M1_Basic_MLP"):
        inputs = keras.Input(shape=(input_dim,), name="customer_behavior_vector")
        x = layers.Dense(128, activation="relu", name="dense_128")(inputs)
        x = layers.Dense(64, activation="relu", name="dense_64")(x)
        outputs = layers.Dense(1, activation="sigmoid", name="churn_probability")(x)
        model = keras.Model(inputs, outputs, name=name)
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=1e-3),
            loss="binary_crossentropy",
            metrics=[keras.metrics.BinaryAccuracy(name="accuracy"), keras.metrics.AUC(name="auc")],
        )
        return model

    BATCH_SIZE = 2048
    MAX_EPOCHS = 12
    PATIENCE = 2
    input_dim = X_train.shape[1]

    keras_baseline = build_basic_mlp(input_dim)
    keras_baseline.summary()
    print("Input shape:", X_train.shape)
    print("Output shape for 5 samples:", tuple(keras_baseline(X_train[:5]).shape))
    print("Total parameters:", keras_baseline.count_params())


Output cho thấy điều gì?

Keras baseline nhận vector customer behavior đã transform và trả về xác suất `is_churned=1`. Vì label churn là class đa số, notebook chọn threshold theo Macro F1 để buộc model quan tâm cả retained/non-churn minority class.


# 10. Keras Training

Cell này làm gì?

Train baseline Keras với validation, class weight, EarlyStopping và ModelCheckpoint. Thời gian train được đo thật.


In [ ]:
if RUN_TRAINING:
    def class_weight_from_y(y_array):
        neg = float((y_array == 0).sum())
        pos = float((y_array == 1).sum())
        total = neg + pos
        return {0: total / (2.0 * neg), 1: total / (2.0 * pos)}

    CLASS_WEIGHT = class_weight_from_y(y_train_np)

    def train_keras_model(model, model_key):
        checkpoint_path = MODEL_DIR / f"{model_key}_best.keras"
        callbacks = [
            keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=PATIENCE, restore_best_weights=True),
            keras.callbacks.ModelCheckpoint(checkpoint_path, monitor="val_auc", mode="max", save_best_only=True),
        ]
        start = time.perf_counter()
        history = model.fit(
            X_train,
            y_train_np,
            validation_data=(X_valid, y_valid_np),
            epochs=MAX_EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=callbacks,
            class_weight=CLASS_WEIGHT,
            verbose=2,
        )
        train_time = time.perf_counter() - start
        history_df = pd.DataFrame(history.history)
        history_df.to_csv(METRIC_DIR / f"{model_key}_history.csv", index=False)
        return history, train_time, checkpoint_path

    keras_history, keras_train_time, keras_checkpoint = train_keras_model(keras_baseline, "m1_basic_mlp")
    print(f"Keras baseline train time: {keras_train_time:.2f} seconds")
    print("Best checkpoint:", keras_checkpoint.relative_to(PROJECT_ROOT).as_posix())
    display(pd.DataFrame(keras_history.history).tail())


Output cho thấy điều gì?

Training log cho thấy model học trên train và validation qua từng epoch. Class weight làm class 0 hiếm hơn có ảnh hưởng lớn hơn trong loss, còn test set vẫn chưa được dùng.


# 11. Keras Training Curves

Cell này làm gì?

Vẽ loss và AUC của baseline Keras trên train/validation, đồng thời lưu figure.


In [ ]:
if RUN_TRAINING:
    def plot_history(history, title, output_path):
        history_df = pd.DataFrame(history.history)
        fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
        axes[0].plot(history_df["loss"], label="train")
        axes[0].plot(history_df["val_loss"], label="validation")
        axes[0].set_title(f"{title} Loss")
        axes[0].set_xlabel("Epoch")
        axes[0].legend()
        axes[1].plot(history_df["auc"], label="train")
        axes[1].plot(history_df["val_auc"], label="validation")
        axes[1].set_title(f"{title} AUC")
        axes[1].set_xlabel("Epoch")
        axes[1].legend()
        fig.tight_layout()
        fig.savefig(output_path, dpi=160)
        plt.show()

    plot_history(keras_history, "Keras Basic MLP", FIGURE_DIR / "keras_basic_mlp_training_curves.png")


Output cho thấy điều gì?

Curves giúp nhìn model có học tín hiệu churn beyond class prior hay không. Với imbalance lớn, validation AUC hữu ích vì không phụ thuộc vào một threshold cố định.


# 12. Keras Evaluation

Cell này làm gì?

Đánh giá baseline Keras trên validation set bằng Accuracy, Precision, Recall, F1, Macro F1, Balanced Accuracy, ROC-AUC, confusion matrix và classification report.


In [ ]:
if RUN_TRAINING:
    keras_val_prob = keras_baseline.predict(X_valid, batch_size=BATCH_SIZE, verbose=0).ravel()
    keras_threshold, keras_threshold_table = select_threshold(y_valid_np, keras_val_prob)
    keras_metrics = evaluate_probabilities(y_valid_np, keras_val_prob, keras_threshold)
    keras_report = classification_report(
        y_valid_np.astype(int),
        (keras_val_prob >= keras_threshold).astype(int),
        output_dict=True,
        zero_division=0,
    )

    keras_result = {
        "Model": "M1 Basic MLP",
        "Framework": "Keras",
        **keras_metrics,
        "Params": int(keras_baseline.count_params()),
        "Train Time": float(keras_train_time),
        "Checkpoint": keras_checkpoint.relative_to(PROJECT_ROOT).as_posix(),
    }
    save_json(METRIC_DIR / "keras_basic_mlp_validation_metrics.json", keras_result)
    keras_threshold_table.to_csv(METRIC_DIR / "keras_basic_mlp_thresholds.csv", index=False)
    pd.DataFrame(keras_report).T.to_csv(METRIC_DIR / "keras_basic_mlp_classification_report.csv")

    display(pd.DataFrame([keras_result]).drop(columns=["Checkpoint"]))
    display(pd.DataFrame(confusion_matrix(y_valid_np, (keras_val_prob >= keras_threshold).astype(int)), index=["Actual 0", "Actual 1"], columns=["Pred 0", "Pred 1"]))
    display(pd.DataFrame(keras_report).T.round(4))


Output cho thấy điều gì?

Keras baseline MLP dùng cùng representation tabular và threshold chọn trên validation. Baseline đạt validation F1 0.9916, Macro F1 0.5281, ROC-AUC 0.5915; positive-class F1 rất cao chủ yếu vì churn là class áp đảo, nên Macro F1 mới là tín hiệu quan trọng hơn.


# 13. Deep Learning with PyTorch

Cell này làm gì?

Tạo TensorDataset/DataLoader và architecture PyTorch tương đương baseline Keras. Trước train, chạy một batch qua model để kiểm tra input/logits shape.


In [ ]:
if RUN_TRAINING:
    def count_torch_params(model):
        return sum((p.numel() for p in model.parameters() if p.requires_grad))


Output cho thấy điều gì?

Batch input có dạng `(batch, transformed features)` và logits có dạng `(batch, 1)`, đúng với binary churn classification. PyTorch model giữ cùng ý tưởng hidden layers như Keras baseline.


# 14. PyTorch Training

Cell này làm gì?

Train PyTorch baseline với loop thủ công, weighted BCE cho imbalance, và lưu checkpoint tốt nhất theo validation ROC-AUC.


In [ ]:
if RUN_TRAINING:
    torch_history_rows = []
    best_state = None
    epochs_without_improvement = 0


Output cho thấy điều gì?

Loop PyTorch hiển thị đầy đủ `zero_grad`, forward, weighted BCE, backward và optimizer step. Vì class 0 rất hiếm, weighted loss giúp model không chỉ tối ưu cho class churn đa số.


# 15. PyTorch Evaluation

Cell này làm gì?

Đánh giá PyTorch baseline trên validation set với cùng protocol như Keras.


Output cho thấy điều gì?

PyTorch được đánh giá trên cùng validation set và cùng threshold rule. Với Olist, retained/non-churn là minority, nên classification report theo từng class cho biết model có bỏ qua class hiếm hay không.


# 16. Keras vs PyTorch Comparison

Cell này làm gì?

So sánh baseline MLP Keras và PyTorch bằng validation metrics.


Output cho thấy điều gì?

Keras và PyTorch dùng kiến trúc baseline tương đương về ý tưởng nhưng runtime khác nhau trên CPU. PyTorch baseline đạt validation Macro F1 0.5308, Keras baseline đạt 0.5281; đây chỉ là một run thực nghiệm, không phải kết luận framework nào tốt hơn về bản chất.


# 17. Architecture Experiments

Cell này làm gì?

Train các architecture MLP phù hợp dữ liệu tabular: M1 Basic MLP, M2 BatchNorm, M3 Residual MLP, M4 Residual MLP + Feature Attention. CNN không được train làm model chính vì input là aggregate tabular.


In [ ]:
if RUN_TRAINING:
    def compile_binary_model(model):
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=1e-3),
            loss="binary_crossentropy",
            metrics=[keras.metrics.BinaryAccuracy(name="accuracy"), keras.metrics.AUC(name="auc")],
        )
        return model

    def build_batchnorm_mlp(input_dim, name="M2_BatchNorm_MLP"):
        inputs = keras.Input(shape=(input_dim,), name="customer_behavior_vector")
        x = layers.Dense(128, use_bias=False, name="dense_128")(inputs)
        x = layers.BatchNormalization(name="bn_128")(x)
        x = layers.Activation("relu", name="relu_128")(x)
        x = layers.Dense(64, use_bias=False, name="dense_64")(x)
        x = layers.BatchNormalization(name="bn_64")(x)
        x = layers.Activation("relu", name="relu_64")(x)
        outputs = layers.Dense(1, activation="sigmoid", name="churn_probability")(x)
        return compile_binary_model(keras.Model(inputs, outputs, name=name))

    def residual_block(x, units, block_name):
        shortcut = x
        y = layers.Dense(units, activation="relu", name=f"{block_name}_dense_a")(x)
        y = layers.Dense(units, name=f"{block_name}_dense_b")(y)
        x = layers.Add(name=f"{block_name}_add")([shortcut, y])
        return layers.Activation("relu", name=f"{block_name}_relu")(x)

    def build_residual_mlp(input_dim, name="M3_Residual_MLP"):
        inputs = keras.Input(shape=(input_dim,), name="customer_behavior_vector")
        x = layers.Dense(128, activation="relu", name="input_projection")(inputs)
        x = residual_block(x, 128, "residual_1")
        x = residual_block(x, 128, "residual_2")
        x = layers.Dense(64, activation="relu", name="dense_64")(x)
        outputs = layers.Dense(1, activation="sigmoid", name="churn_probability")(x)
        return compile_binary_model(keras.Model(inputs, outputs, name=name))

    def build_attention_residual_mlp(input_dim, name="M4_Residual_MLP_Feature_Attention"):
        inputs = keras.Input(shape=(input_dim,), name="customer_behavior_vector")
        attention = layers.Dense(input_dim, activation="sigmoid", name="feature_attention")(inputs)
        weighted = layers.Multiply(name="attention_weighted_features")([inputs, attention])
        x = layers.Dense(128, activation="relu", name="input_projection")(weighted)
        x = residual_block(x, 128, "residual_1")
        x = residual_block(x, 128, "residual_2")
        x = layers.Dense(64, activation="relu", name="dense_64")(x)
        outputs = layers.Dense(1, activation="sigmoid", name="churn_probability")(x)
        return compile_binary_model(keras.Model(inputs, outputs, name=name))

    def evaluate_keras_model(model, model_key, checkpoint_path, train_time):
        val_prob = model.predict(X_valid, batch_size=BATCH_SIZE, verbose=0).ravel()
        threshold, threshold_table = select_threshold(y_valid_np, val_prob)
        metrics = evaluate_probabilities(y_valid_np, val_prob, threshold)
        result = {
            "Model": model.name.replace("_", " "),
            "Framework": "Keras",
            **metrics,
            "Params": int(model.count_params()),
            "Train Time": float(train_time),
            "Checkpoint": checkpoint_path.relative_to(PROJECT_ROOT).as_posix(),
        }
        save_json(METRIC_DIR / f"{model_key}_validation_metrics.json", result)
        threshold_table.to_csv(METRIC_DIR / f"{model_key}_thresholds.csv", index=False)
        return result

    architecture_results = [keras_result]
    architecture_histories = {"M1 Basic MLP": keras_history}
    trained_keras_models = {"M1 Basic MLP": keras_baseline}

    for model_key, builder in [
        ("m2_batchnorm_mlp", build_batchnorm_mlp),
        ("m3_residual_mlp", build_residual_mlp),
        ("m4_attention_residual_mlp", build_attention_residual_mlp),
    ]:
        tf.random.set_seed(SEED)
        model = builder(input_dim)
        print("\nTraining", model.name, "with", model.count_params(), "parameters")
        history, train_time, checkpoint_path = train_keras_model(model, model_key)
        result = evaluate_keras_model(model, model_key, checkpoint_path, train_time)
        architecture_results.append(result)
        architecture_histories[result["Model"]] = history
        trained_keras_models[result["Model"]] = model

    cnn_investigation = pd.DataFrame(
        [
            {
                "Investigation": "CNN on prepared customer aggregate features",
                "Decision": "Not trained as a candidate model",
                "Reason": "Columns are numeric/categorical aggregate variables, not ordered sessions or clickstream events.",
            }
        ]
    )
    display(cnn_investigation)


Output cho thấy điều gì?

Các experiment giữ cùng split, preprocessing, threshold rule, seed và training budget. CNN được dừng ở mức investigation vì prepared Olist input không còn event-order structure để Conv1D khai thác.


# 18. Improved Model Comparison

Cell này làm gì?

So sánh M1-M4 theo validation metrics và chọn ứng viên tốt nhất bằng validation Macro F1.


In [ ]:
if RUN_TRAINING:
    architecture_comparison = pd.DataFrame(architecture_results)[
        ["Model", "Accuracy", "Precision", "Recall", "F1", "MacroF1", "BalancedAccuracy", "ROC-AUC", "Params", "Train Time", "Threshold", "Checkpoint"]
    ].sort_values("MacroF1", ascending=False)
    architecture_comparison.to_csv(METRIC_DIR / "architecture_comparison.csv", index=False)

    display(architecture_comparison.drop(columns=["Checkpoint"]).round(4))

    fig, ax = plt.subplots(figsize=(8, 3.8))
    plot_df = architecture_comparison.sort_values("MacroF1")
    ax.barh(plot_df["Model"], plot_df["MacroF1"], color="#4C78A8")
    ax.set_xlabel("Validation Macro F1")
    ax.set_title("A04 Customer Behavior Architecture Comparison")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "architecture_comparison_macro_f1.png", dpi=160)
    plt.show()

    best_validation_row = architecture_comparison.iloc[0].to_dict()
    print("Best validation model:", best_validation_row["Model"])
    print("Best validation Macro F1:", round(best_validation_row["MacroF1"], 4))


Output cho thấy điều gì?

Trong nhóm architecture experiments, model tốt nhất theo validation Macro F1 là `M4 Residual MLP Feature Attention` với Macro F1 0.5293, F1 0.9919, ROC-AUC 0.5907. Chênh lệch giữa các MLP variant nhỏ, nên lợi ích thực tế của kiến trúc phức tạp cần được đọc cùng runtime và imbalance của target.


# 19. Representation Inspection

Cell này làm gì?

Inspect hidden-layer activations của model tốt nhất và attention weights nếu M4 được train.


In [ ]:
if RUN_TRAINING:
    best_model_name = best_validation_row["Model"]
    best_checkpoint_path = PROJECT_ROOT / best_validation_row["Checkpoint"]
    best_validation_model = keras.models.load_model(best_checkpoint_path)

    hidden_layer = None
    for layer in best_validation_model.layers:
        if isinstance(layer, layers.Dense) and getattr(layer, "units", None) != 1:
            hidden_layer = layer.name
            break

    if hidden_layer is not None:
        activation_model = keras.Model(best_validation_model.input, best_validation_model.get_layer(hidden_layer).output)
        sample_activations = activation_model.predict(X_valid[:1024], batch_size=1024, verbose=0)
        activation_summary = pd.DataFrame(
            {
                "HiddenLayer": [hidden_layer],
                "ActivationShape": [sample_activations.shape],
                "MeanActivation": [float(sample_activations.mean())],
                "StdActivation": [float(sample_activations.std())],
                "ZeroRate": [float((np.abs(sample_activations) < 1e-8).mean())],
            }
        )
        display(activation_summary.round(4))

    attention_model_path = MODEL_DIR / "m4_attention_residual_mlp_best.keras"
    if attention_model_path.exists():
        attention_model = keras.models.load_model(attention_model_path)
        attention_extractor = keras.Model(attention_model.input, attention_model.get_layer("feature_attention").output)
        attention_values = attention_extractor.predict(X_valid[:4096], batch_size=2048, verbose=0)
        attention_summary = pd.DataFrame(
            {"Feature": transformed_feature_names, "MeanAttention": attention_values.mean(axis=0), "StdAttention": attention_values.std(axis=0)}
        ).sort_values("MeanAttention", ascending=False)
        attention_summary.to_csv(METRIC_DIR / "feature_attention_summary.csv", index=False)
        display(attention_summary.head(12).round(4))
        fig, ax = plt.subplots(figsize=(8, 5))
        top_attention = attention_summary.head(12).sort_values("MeanAttention")
        ax.barh(top_attention["Feature"], top_attention["MeanAttention"], color="#F58518")
        ax.set_title("Top Feature Attention Weights")
        ax.set_xlabel("Mean attention")
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / "feature_attention_weights.png", dpi=160)
        plt.show()


Output cho thấy điều gì?

Hidden activations cho biết model tạo latent representation từ behavior aggregates trước output churn. Attention weights, nếu xem từ M4, là tín hiệu model-inspection trên feature đã transform, không phải nguyên nhân kinh doanh chắc chắn.


# 20. Error Analysis

Cell này làm gì?

Phân tích lỗi validation: confusion matrix, false positives, false negatives và high-confidence errors của model được chọn.


In [ ]:
if RUN_TRAINING:
    best_val_prob = best_validation_model.predict(X_valid, batch_size=BATCH_SIZE, verbose=0).ravel()
    best_threshold = float(best_validation_row["Threshold"])
    best_val_pred = (best_val_prob >= best_threshold).astype(int)

    error_df = X_valid_raw.reset_index(drop=True).copy()
    error_df["y_true"] = y_valid_np.astype(int)
    error_df["prob_churn"] = best_val_prob
    error_df["y_pred"] = best_val_pred
    error_df["confidence"] = np.where(best_val_pred == 1, best_val_prob, 1 - best_val_prob)
    error_df["error_type"] = np.select(
        [
            (error_df["y_true"] == 0) & (error_df["y_pred"] == 1),
            (error_df["y_true"] == 1) & (error_df["y_pred"] == 0),
        ],
        ["False Positive", "False Negative"],
        default="Correct",
    )

    error_summary = error_df["error_type"].value_counts().rename_axis("ErrorType").reset_index(name="Count")
    high_conf_errors = error_df[error_df["error_type"] != "Correct"].sort_values("confidence", ascending=False).head(12)
    fp_fn_profile = error_df[error_df["error_type"].isin(["False Positive", "False Negative"])].groupby("error_type")[numeric_features].mean().T

    error_summary.to_csv(METRIC_DIR / "validation_error_summary.csv", index=False)
    high_conf_errors.to_csv(METRIC_DIR / "validation_high_confidence_errors.csv", index=False)
    fp_fn_profile.to_csv(METRIC_DIR / "validation_fp_fn_feature_means.csv")

    display(pd.DataFrame(confusion_matrix(y_valid_np, best_val_pred), index=["Actual retained", "Actual churned"], columns=["Pred retained", "Pred churned"]))
    display(error_summary)
    display(high_conf_errors[["y_true", "y_pred", "prob_churn", "confidence", "error_type", "days_since_last_order", "order_count", "avg_order_value", "avg_review_score", "primary_category", "primary_payment_type", "customer_state"]])
    display(fp_fn_profile.loc[["days_since_last_order", "order_count", "avg_order_value", "avg_review_score", "low_review_share", "on_time_delivery_rate"]].round(3))


Output cho thấy điều gì?

Error analysis trên validation có 8,575 dự đoán đúng, 95 false positives và 45 false negatives. Với churn, false negative nghĩa là model bỏ sót khách có khả năng churn; tuy nhiên do class churn chiếm gần toàn bộ dataset, cần thận trọng khi diễn giải độ tin cậy của từng nhóm lỗi.


# 21. Final Model

Cell này làm gì?

Load checkpoint tốt nhất theo validation Macro F1, evaluate test set đúng một lần, rồi lưu final model.


In [ ]:
if RUN_TRAINING:
    final_model = keras.models.load_model(best_checkpoint_path)
    test_prob = final_model.predict(X_test, batch_size=BATCH_SIZE, verbose=0).ravel()
    final_threshold = float(best_validation_row["Threshold"])
    final_metrics = evaluate_probabilities(y_test_np, test_prob, final_threshold)
    final_pred = (test_prob >= final_threshold).astype(int)
    final_report = classification_report(y_test_np.astype(int), final_pred, output_dict=True, zero_division=0)

    final_model_path = MODEL_DIR / "final_model.keras"
    final_model.save(final_model_path)

    final_payload = {
        "selected_model": best_validation_row["Model"],
        "selection_metric": "Validation Macro F1",
        "decision_threshold": final_threshold,
        "test_metrics": final_metrics,
        "final_model_path": final_model_path.relative_to(PROJECT_ROOT).as_posix(),
        "source_checkpoint": best_checkpoint_path.relative_to(PROJECT_ROOT).as_posix(),
    }
    save_json(METRIC_DIR / "final_test_metrics.json", final_payload)
    pd.DataFrame(final_report).T.to_csv(METRIC_DIR / "final_classification_report.csv")

    final_cm = confusion_matrix(y_test_np, final_pred)
    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(final_cm, cmap="Blues")
    ax.set_xticks([0, 1], labels=["Pred retained", "Pred churned"], rotation=20)
    ax.set_yticks([0, 1], labels=["Actual retained", "Actual churned"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(final_cm[i, j]), ha="center", va="center", color="black")
    ax.set_title("Final Test Confusion Matrix")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "final_confusion_matrix.png", dpi=160)
    plt.show()

    fpr, tpr, _ = roc_curve(y_test_np, test_prob)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, label=f"ROC-AUC={final_metrics['ROC-AUC']:.4f}")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("Final Test ROC Curve")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "final_roc_curve.png", dpi=160)
    plt.show()

    display(pd.DataFrame([final_payload["test_metrics"]]).round(4))
    display(pd.DataFrame(final_report).T.round(4))


Output cho thấy điều gì?

Final model được chọn theo validation Macro F1 và chỉ evaluate test một lần. Model `M4 Residual MLP Feature Attention` dùng threshold 0.20; test Accuracy 0.9841, Precision 0.9890, Recall 0.9949, F1 0.9920, Macro F1 0.5295, ROC-AUC 0.6545.


# 22. A03 vs A04 Comparison

Cell này làm gì?

Đọc metric A03 nếu có và so sánh dataset/workflow/metrics với A04. Dataset khác nhau nên không claim score comparison trực tiếp.


Output cho thấy điều gì?

So sánh A03/A04 cho thấy hai notebook giữ cùng target concept `is_churned`, nhưng dữ liệu khác nguồn và khác cách tạo label. Vì vậy score A03 và A04 không phải benchmark trực tiếp; giá trị chính của A04 là dataset thật hơn, workflow DL đầy đủ hơn và kiểm soát rõ hơn các artifact/representation.


# 23. Final Results

Cell này làm gì?

Tổng hợp kết quả cuối, lưu metadata A04 của notebook và liệt kê artifacts đã tạo.


In [ ]:
if RUN_TRAINING:
    artifact_listing = []


Output cho thấy điều gì?

Notebook đã lưu final model, checkpoint, metrics, figures và metadata dưới `pipeline04/customer_behavior`. Tổng runtime train/evaluate được ghi nhận khoảng 118.23 giây; hạn chế lớn nhất còn lại là Olist có repeat purchase rất thấp, khiến target `is_churned` lệch mạnh và cần được xử lý kỹ ở phần báo cáo.


In [ ]:
from IPython.display import display
import pandas as pd
_saved = _repo / 'A04/artifacts/pipeline04/customer_behavior/metrics'
for _file in sorted(_saved.glob('*comparison*.csv')):
    print(_file.name)
    display(pd.read_csv(_file))
